# Real-World Examples Using the `src` Package

This notebook demonstrates practical applications of the core statistical modules on realistic business and research scenarios.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import bootstrap
from src import hypothesis_testing
from src import empirical_bayes
from src import shrinkage
from src import lasso
from src import survival

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

---

## Scenario 1: E-commerce — Bootstrap Confidence Interval for Average Order Value

**Business Question**: What is the 95% confidence interval for the true average order value?

In [ ]:
# Simulate realistic order values (right-skewed)
order_values = np.random.lognormal(mean=4.5, sigma=0.6, size=500)

def mean_stat(x): return np.mean(x)

ci_low, ci_high = bootstrap.bootstrap_confidence_interval(
    order_values, mean_stat, n_bootstrap=10000, confidence_level=0.95
)

print(f"Sample Mean Order Value: {np.mean(order_values):.2f}")
print(f"Bootstrap 95% CI: [{ci_low:.2f}, {ci_high:.2f}]")

---

## Scenario 2: A/B Test — Permutation Test for Conversion Rate

**Business Question**: Is there a statistically significant difference in conversion rate between two website versions?

In [ ]:
# Simulate conversion data (binary)
conversions_A = np.random.binomial(1, 0.12, 1200)   # Control
conversions_B = np.random.binomial(1, 0.15, 1150)   # Treatment

obs_diff, p_value = hypothesis_testing.permutation_test(
    conversions_A, conversions_B, statistic=np.mean, n_permutations=5000
)

print(f"Observed difference in conversion rate: {obs_diff*100:.2f}%")
print(f"Permutation test p-value: {p_value:.4f}")
print("Significant at 5% level?", p_value < 0.05)

---

## Scenario 3: Retail Performance — James-Stein Shrinkage for Store Sales

**Business Question**: Improve estimates of true sales performance across many stores by shrinking noisy observations.

In [ ]:
# Simulate sales data from many stores
true_sales = np.random.normal(85000, 12000, 25)
observed_sales = true_sales + np.random.normal(0, 15000, 25)  # noisy observations

shrunk_sales = empirical_bayes.james_stein_estimator(observed_sales)

plt.figure(figsize=(10,5))
plt.plot(observed_sales, 'o', label='Observed Sales', alpha=0.6)
plt.plot(shrunk_sales, 's', label='James-Stein Shrunk', markersize=8)
plt.axhline(np.mean(observed_sales), color='gray', linestyle='--', label='Grand Mean')
plt.legend()
plt.title('Shrinkage of Store Sales Estimates')
plt.xlabel('Store Index')
plt.ylabel('Sales ($)')
plt.show()

---

## Scenario 4: Customer Churn Prediction — LASSO Feature Selection

**Business Question**: Which customer features are most predictive of churn?

In [ ]:
# Simulate customer features
n_customers = 800
X = np.random.randn(n_customers, 12)
true_important = [3, 0, -2.5, 0, 1.8, 0, 0, -1.5, 0, 2.2, 0, 0]
churn_prob = 1 / (1 + np.exp(-(X @ true_important)))
y = (np.random.rand(n_customers) < churn_prob).astype(int)

coef, _ = lasso.fit_lasso_sklearn(X, y, alpha=0.3)

feature_names = [f'Feature_{i}' for i in range(12)]
selected = pd.DataFrame({'Feature': feature_names, 'Coefficient': coef})
selected = selected[selected.Coefficient != 0].sort_values('Coefficient', key=abs, ascending=False)

print("Top predictive features for churn:")
print(selected)

---

## Scenario 5: Subscription Business — Survival Analysis for Customer Lifetime

**Business Question**: What is the median customer lifetime, and is there a difference between two customer segments?

In [ ]:
# Simulate subscription durations (days)
np.random.seed(42)
durations_premium = np.random.exponential(180, 300)      # Premium customers
events_premium    = np.random.binomial(1, 0.75, 300)

durations_standard = np.random.exponential(120, 450)     # Standard customers
events_standard    = np.random.binomial(1, 0.82, 450)

median_premium = survival.kaplan_meier_median_survival(durations_premium, events_premium)
median_standard = survival.kaplan_meier_median_survival(durations_standard, events_standard)

print(f"Median lifetime - Premium: {median_premium:.0f} days")
print(f"Median lifetime - Standard: {median_standard:.0f} days")

# Log-rank test
stat, p_val = survival.log_rank_test(durations_premium, events_premium, durations_standard, events_standard)
print(f"Log-rank test p-value: {p_val:.4f}")

---

## Summary

This notebook showed how the `src` package can be applied to real business problems:

- **Bootstrap** → Uncertainty quantification for business metrics
- **Hypothesis Testing** → Rigorous A/B testing
- **Shrinkage** → Improving estimates across many units (stores, products, etc.)
- **LASSO** → Interpretable feature selection
- **Survival Analysis** → Customer lifetime and churn modeling

These modules provide a strong foundation for statistical work in production environments.